# Approach 4: XLM-RoBERTa Fine-tuning for Multi-label Intent Classification

**Model:** `facebook/xlm-roberta-base` — pre-trained on text from 100 languages including both Vietnamese and English.

**Core argument:** The customer chat data is code-switched Vietnamese-English (`lego`, `ferrari`, `ship` embedded in Vietnamese). XLM-RoBERTa is the only model in this comparison pre-trained on **both** languages simultaneously, giving it full representations for English brand names and Vietnamese vocabulary in the same embedding space.

| Component | Choice | Reason |
|---|---|---|
| Teen code normalization | ✅ Applied | Reduces vocabulary fragmentation from informal abbreviations |
| Multi-label augmentation | ✅ Applied | Improves co-occurring intent detection |
| Random token masking | ✅ Applied | Reduces over-reliance on specific trigger words |
| Handcrafted features | ❌ Skipped | BERT handles these contextually |
| Word segmentation | ❌ Skipped | XLM-RoBERTa's SentencePiece tokenizer handles mixed-language text natively |

**Key comparison:** Approach 4 vs Approach 2 (ViSoBERT) tests cross-lingual pretraining vs social-media pretraining for code-switched Vietnamese chat. Approach 4 vs Approach 3 (PhoBERT) tests whether a multilingual model handles informal text better than a monolingual model with explicit normalization.

**Prerequisite:** Run `data_preparation/data_split.ipynb` first.

In [ ]:
# Requirements for this notebook
# See requirements.txt at the project root for the full list
!pip install torch transformers scikit-learn numpy pandas matplotlib seaborn joblib --quiet
!pip install hf_xet --quiet  # faster Hugging Face downloads

In [ ]:
import json
import os
import re
import warnings
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import f1_score, hamming_loss, accuracy_score, classification_report

warnings.filterwarnings('ignore')
os.makedirs('results', exist_ok=True)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR   = '../final_data'
MODEL_NAME = 'xlm-roberta-base'   # same model as facebook/xlm-roberta-base

print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')

## 1. Load Pre-split Data

In [ ]:
def load_jsonl(path):
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            texts.append(item['text'])
            labels.append(item['cats'])
    return texts, labels

train_texts, train_labels = load_jsonl(f'{DATA_DIR}/train.jsonl')
val_texts,   val_labels   = load_jsonl(f'{DATA_DIR}/val.jsonl')
test_texts,  test_labels  = load_jsonl(f'{DATA_DIR}/test.jsonl')

mlb     = joblib.load(f'{DATA_DIR}/mlb.joblib')
y_train = mlb.transform(train_labels).astype(np.float32)
y_val   = mlb.transform(val_labels).astype(np.float32)
y_test  = mlb.transform(test_labels).astype(np.float32)

print(f'Train : {len(train_texts):4d} samples')
print(f'Val   : {len(val_texts):4d} samples')
print(f'Test  : {len(test_texts):4d} samples')
print(f'Labels: {len(mlb.classes_)} classes')

## 2. Teen Code Normalization

Same normalization pipeline as Approach 3.

English tokens matching `/^[a-z0-9-]+$/` that are **not** in the teen code dictionary are protected — brand names (`lego`, `ferrari`, `ship`) pass through unchanged, which is important for XLM-RoBERTa since it has strong pre-trained representations for these English words.

In [ ]:
TEEN_CODE = {
    'k'    : 'không',  'ko'   : 'không',  'kh'   : 'không',
    'khum' : 'không',  'hum'  : 'không',  'kum'  : 'không',
    'đc'   : 'được',   'dc'   : 'được',
    'vs'   : 'với',
    'b'    : 'bạn',    'm'    : 'mình',
    'sốp'  : 'shop',   'sốc'  : 'shop',
    'ck'   : 'chuyển khoản',
    'rep'  : 'trả lời',
    'sp'   : 'sản phẩm',
    'mn'   : 'mọi người',
    'r'    : 'rồi',    'rui'  : 'rồi',
    'nha'  : 'nhé',    'nhen' : 'nhé',
}

def normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    tokens = text.split()
    result = []
    for token in tokens:
        if re.match(r'^[a-z0-9\-]+$', token) and token not in TEEN_CODE:
            result.append(token)  # protect English tokens
        else:
            result.append(TEEN_CODE.get(token, token))
    return ' '.join(result)

# Verify normalization + English protection
examples = [
    'sốp ơi còn hàng k ạ',
    'shop còn hàng con ferrari ko ạ',
    'cho e đặt nha, ck dc kh ạ',
    'sp lego city giá bao nhiêu ạ',
]
for text in examples:
    print(f'Before: {text}')
    print(f'After : {normalize(text)}')
    print()

In [ ]:
norm_train = [normalize(t) for t in train_texts]
norm_val   = [normalize(t) for t in val_texts]
norm_test  = [normalize(t) for t in test_texts]
print('Normalization applied to all splits.')

## 3. Multi-label Augmentation

Duplicates multi-label training samples 2× — consistent with Approaches 1, 2, and 3.  
Applied to the **training set only**.

In [ ]:
multi_idx    = [i for i, l in enumerate(train_labels) if len(l) > 1]
extra_texts  = [norm_train[i]  for i in multi_idx] * 2
extra_labels = [train_labels[i] for i in multi_idx] * 2

aug_texts  = norm_train + extra_texts
aug_labels = train_labels + extra_labels

print(f'Original train : {len(norm_train)}')
print(f'Multi-label    : {len(multi_idx)}')
print(f'Augmented total: {len(aug_texts)}')

## 4. Random Token Masking

Randomly drops 15% of tokens per sample, creating one additional masked copy of the augmented training set.  
Val and test texts are never masked.

In [ ]:
MASK_PROB = 0.15

def random_mask(text: str, mask_prob: float = MASK_PROB, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    tokens = text.split()
    kept = [t for t in tokens if rng.random() > mask_prob]
    return ' '.join(kept) if kept else text

rng = np.random.default_rng(42)
masked_texts = [random_mask(t, rng=rng) for t in aug_texts]

masked_train_texts  = aug_texts + masked_texts
masked_train_labels = aug_labels + aug_labels
y_masked = mlb.transform(masked_train_labels).astype(np.float32)

print(f'Augmented train : {len(aug_texts)}')
print(f'After masking   : {len(masked_train_texts)} (1 masked copy added)')
print(f'\nExample:')
print(f'  Original: {aug_texts[0]}')
print(f'  Masked  : {masked_texts[0]}')

## 5. Tokenizer and DataLoaders

XLM-RoBERTa uses a SentencePiece tokenizer trained on 100 languages. It handles Vietnamese and English tokens natively in the same vocabulary — English brand names like `lego` and `ferrari` have full pre-trained representations.

In [ ]:
MAX_LEN    = 128
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size: {tokenizer.vocab_size:,}')

class IntentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    encoding = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors='pt',
    )
    return encoding, torch.stack(labels)

train_ds = IntentDataset(masked_train_texts, y_masked)
val_ds   = IntentDataset(norm_val,           y_val)
test_ds  = IntentDataset(norm_test,          y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
print(f'Test batches : {len(test_loader)}')

## 6. Model

XLM-RoBERTa-base + linear classification head:
```
Normalized text → XLM-RoBERTa → [CLS] (768-dim) → Dropout(0.3) → Linear(768→32)
```

XLM-RoBERTa-base has 278M parameters — about 2× larger than PhoBERT-base (135M). `classifier_dropout=0.3` compensates for the small dataset size relative to the model capacity.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(mlb.classes_),
    problem_type='multi_label_classification',
    classifier_dropout=0.3,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 7. Training

| Hyperparameter | Value |
|---|---|
| Learning rate | 2e-5 |
| Max epochs | 20 (early stopping patience = 5) |
| Warmup | 10% of total steps |
| Gradient clipping | 1.0 |
| Dropout on CLS | 0.3 |

In [ ]:
EPOCHS       = 20
LR           = 2e-5
WARMUP_RATIO = 0.1
PATIENCE     = 5

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Total steps  : {total_steps}')
print(f'Warmup steps : {warmup_steps}')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for encoding, labels in loader:
        encoding = {k: v.to(device) for k, v in encoding.items()}
        labels   = labels.to(device)
        outputs  = model(**encoding, labels=labels)
        loss     = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device, thresholds=0.5):
    model.eval()
    all_proba, all_labels = [], []
    with torch.no_grad():
        for encoding, labels in loader:
            encoding = {k: v.to(device) for k, v in encoding.items()}
            logits   = model(**encoding).logits
            proba    = torch.sigmoid(logits).cpu().numpy()
            all_proba.append(proba)
            all_labels.append(labels.numpy())
    all_proba  = np.vstack(all_proba)
    all_labels = np.vstack(all_labels)
    preds      = (all_proba >= thresholds).astype(int)
    macro_f1   = f1_score(all_labels, preds, average='macro', zero_division=0)
    return macro_f1, all_proba, all_labels

In [ ]:
train_losses  = []
val_macro_f1s = []
best_val_f1   = 0.0
best_epoch    = 0
no_improve    = 0

for epoch in range(1, EPOCHS + 1):
    train_loss   = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
    val_f1, _, _ = evaluate(model, val_loader, DEVICE, thresholds=0.5)

    train_losses.append(train_loss)
    val_macro_f1s.append(val_f1)

    improved = val_f1 > best_val_f1
    if improved:
        best_val_f1 = val_f1
        best_epoch  = epoch
        no_improve  = 0
        torch.save(model.state_dict(), 'results/best_model.pt')
    else:
        no_improve += 1

    flag = ' *saved*' if improved else ''
    print(f'Epoch {epoch:2d}/{EPOCHS}  |  Train Loss: {train_loss:.4f}  |  Val Macro-F1: {val_f1:.4f}{flag}')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).')
        break

print(f'\nBest Val Macro-F1: {best_val_f1:.4f} at epoch {best_epoch}')

## 8. Training Curves

Train loss (left axis) and Val Macro-F1 (right axis) per epoch.  
**Note:** F1 may stay at 0.0 for the first 2–4 epochs — this is normal for multi-label classification with class imbalance. The model is learning (loss drops), but sigmoid outputs need several epochs before consistently crossing the 0.5 threshold.

In [ ]:
epochs_ran = range(1, len(train_losses) + 1)

fig, ax1 = plt.subplots(figsize=(9, 5))
color_loss, color_f1 = '#ee854a', '#4878d0'

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train Loss', color=color_loss)
ax1.plot(epochs_ran, train_losses, 'o-', color=color_loss, label='Train Loss')
ax1.tick_params(axis='y', labelcolor=color_loss)
ax1.set_xticks(list(epochs_ran))

ax2 = ax1.twinx()
ax2.set_ylabel('Val Macro-F1', color=color_f1)
ax2.plot(epochs_ran, val_macro_f1s, 's--', color=color_f1, label='Val Macro-F1')
ax2.tick_params(axis='y', labelcolor=color_f1)
ax2.axvline(best_epoch, color='gray', linestyle=':', linewidth=1.5,
            label=f'Best epoch ({best_epoch})')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.title('Training Curves — Approach 4: XLM-RoBERTa')
fig.tight_layout()
plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Per-class Threshold Tuning

Load the best checkpoint, collect sigmoid probabilities on the validation set, then find the threshold per class that maximises F1.

In [ ]:
model.load_state_dict(torch.load('results/best_model.pt', map_location=DEVICE))
model.eval()

_, val_proba, val_true = evaluate(model, val_loader, DEVICE, thresholds=0.5)

thresholds = []
for i, cls in enumerate(mlb.classes_):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.95, 0.05):
        preds = (val_proba[:, i] >= t).astype(int)
        f1    = f1_score(val_true[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    thresholds.append(best_t)

df_thresh = pd.DataFrame({'Class': mlb.classes_, 'Threshold': thresholds})
print(df_thresh.to_string(index=False))

## 10. Evaluate on Test Set

In [ ]:
_, test_proba, test_true = evaluate(model, test_loader, DEVICE, thresholds=0.5)
y_pred = (test_proba >= np.array(thresholds)).astype(int)

macro_f1   = f1_score(test_true, y_pred, average='macro',  zero_division=0)
micro_f1   = f1_score(test_true, y_pred, average='micro',  zero_division=0)
h_loss     = hamming_loss(test_true, y_pred)
subset_acc = accuracy_score(test_true, y_pred)

print('=' * 50)
print('TEST RESULTS — Approach 4: XLM-RoBERTa')
print('=' * 50)
print(f'Macro-F1        : {macro_f1:.4f}')
print(f'Micro-F1        : {micro_f1:.4f}')
print(f'Hamming Loss    : {h_loss:.4f}')
print(f'Subset Accuracy : {subset_acc:.4f}')

In [ ]:
print(classification_report(test_true, y_pred, target_names=mlb.classes_, zero_division=0))

In [ ]:
per_class_f1 = f1_score(test_true, y_pred, average=None, zero_division=0)
df_f1 = pd.DataFrame({'Label': mlb.classes_, 'F1': per_class_f1}).sort_values('F1', ascending=True)

colors = ['#d62728' if f < 0.5 else '#ff7f0e' if f < 0.7 else '#2ca02c' for f in df_f1['F1']]
plt.figure(figsize=(12, 10))
plt.barh(df_f1['Label'], df_f1['F1'], color=colors)
plt.axvline(macro_f1, color='steelblue', linestyle='--', linewidth=1.5,
            label=f'Macro-F1 = {macro_f1:.3f}')
plt.xlabel('F1 Score')
plt.title('Per-class F1 — Approach 4: XLM-RoBERTa')
plt.legend()
plt.tight_layout()
plt.savefig('results/per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Lowest F1 : {df_f1.iloc[0]['Label']}  ({df_f1.iloc[0]['F1']:.3f})")
print(f"Highest F1: {df_f1.iloc[-1]['Label']} ({df_f1.iloc[-1]['F1']:.3f})")

## 11. Save Model and Results

In [ ]:
tokenizer.save_pretrained('results/tokenizer')

metrics = {
    'approach'          : 'XLM-RoBERTa',
    'model_name'        : MODEL_NAME,
    'best_epoch'        : best_epoch,
    'best_val_macro_f1' : round(best_val_f1, 4),
    'thresholds'        : {k: round(float(v), 4) for k, v in zip(mlb.classes_, thresholds)},
    'macro_f1'          : round(macro_f1,   4),
    'micro_f1'          : round(micro_f1,   4),
    'hamming_loss'      : round(h_loss,     4),
    'subset_accuracy'   : round(subset_acc, 4),
    'per_class_f1'      : {k: round(float(v), 4) for k, v in zip(mlb.classes_, per_class_f1)},
    'train_losses'      : [round(x, 4) for x in train_losses],
    'val_macro_f1s'     : [round(x, 4) for x in val_macro_f1s],
}

with open('results/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('Saved to approach4/results/:')
for name in ['best_model.pt', 'tokenizer/', 'metrics.json',
             'training_curves.png', 'per_class_f1.png']:
    print(f'  {name}')

## 12. Test Custom Input

Reloads the best checkpoint from `results/` — works after a kernel restart.  
Input is normalized with the same pipeline used during training.

In [ ]:
import json, re, datetime
import numpy as np
import torch
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification

_device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_tokenizer = AutoTokenizer.from_pretrained('results/tokenizer')
_mlb       = joblib.load('../final_data/mlb.joblib')

_model = AutoModelForSequenceClassification.from_pretrained(
    'xlm-roberta-base',   # same model as facebook/xlm-roberta-base
    num_labels=len(_mlb.classes_),
    problem_type='multi_label_classification',
    classifier_dropout=0.3,
    ignore_mismatched_sizes=True,
)
_model.load_state_dict(torch.load('results/best_model.pt', map_location=_device))
_model = _model.to(_device)
_model.eval()

with open('results/metrics.json', encoding='utf-8') as f:
    _thresholds = np.array(list(json.load(f)['thresholds'].values()))

_prediction_log = []

def predict(text: str, top_k: int = 3):
    norm = normalize(text)
    enc  = _tokenizer(
        norm, truncation=True, padding=True,
        max_length=128, return_tensors='pt'
    )
    enc = {k: v.to(_device) for k, v in enc.items()}

    with torch.no_grad():
        logits = _model(**enc).logits
    proba     = torch.sigmoid(logits).cpu().numpy()[0]
    predicted = tuple(cls for cls, p, t in zip(_mlb.classes_, proba, _thresholds) if p >= t)
    ranked    = sorted(zip(_mlb.classes_, proba), key=lambda x: -x[1])

    _prediction_log.append({
        'timestamp' : datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'input'     : text,
        'normalized': norm,
        'predicted' : list(predicted),
        'top_proba' : {label: round(float(p), 4) for label, p in ranked[:top_k]},
    })

    print(f'Input     : {text}')
    print(f'Normalized: {norm}')
    print(f'Predicted : {list(predicted) if predicted else "[none above threshold]"}')
    print(f'Top {top_k} probabilities:')
    for label, prob in ranked[:top_k]:
        marker = ' <--' if label in predicted else ''
        print(f'  {label:<35} {prob:.3f}{marker}')

print('predict() ready.  History log is in _prediction_log.')

In [ ]:
# ── Change this text and re-run ──────────────────────────────────────────────
predict('sốp ơi còn hàng con ferrari k ạ')

In [ ]:
samples = [
    'shop ơi',
    'bao g có hàng thế ạ',
    'cho e đặt màu xanh với ạ',
    'giá con này bao nhiêu vậy shop',
]
for text in samples:
    print('-' * 55)
    predict(text, top_k=2)
print('-' * 55)

## 13. Prediction History Log

In [ ]:
def show_log(n=None):
    entries = _prediction_log if n is None else _prediction_log[-n:]
    if not entries:
        print('Log is empty — run predict() first.')
        return
    print(f'{"#":<4} {"Timestamp":<20} {"Input":<40} {"Predicted"}')
    print('-' * 100)
    for i, e in enumerate(entries):
        idx   = len(_prediction_log) - len(entries) + i + 1
        inp   = e['input'][:38] + '..' if len(e['input']) > 40 else e['input']
        preds = ', '.join(e['predicted']) if e['predicted'] else '[none]'
        print(f'{idx:<4} {e["timestamp"]:<20} {inp:<40} {preds}')
    print(f'\nTotal entries: {len(_prediction_log)}')

show_log()

In [ ]:
LOG_PATH = 'results/prediction_log.jsonl'

def save_log(path=LOG_PATH, mode='append'):
    if not _prediction_log:
        print('Nothing to save.')
        return
    if mode == 'append':
        existing = set()
        try:
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    e = json.loads(line)
                    existing.add((e['timestamp'], e['input']))
        except FileNotFoundError:
            pass
        new = [e for e in _prediction_log if (e['timestamp'], e['input']) not in existing]
        with open(path, 'a', encoding='utf-8') as f:
            for e in new:
                json.dump(e, f, ensure_ascii=False)
                f.write('\n')
        print(f'Appended {len(new)} entries → {path}')
    else:
        with open(path, 'w', encoding='utf-8') as f:
            for e in _prediction_log:
                json.dump(e, f, ensure_ascii=False)
                f.write('\n')
        print(f'Saved {len(_prediction_log)} entries → {path}')

def clear_log():
    count = len(_prediction_log)
    _prediction_log.clear()
    print(f'Cleared {count} entries.')

print('save_log() / clear_log() ready.')